# Text Generation with RNNs — NumPy vs TensorFlow vs PyTorch

This notebook trains a stacked (multi-layer) recurrent neural network three
ways and compares them on the **same data, architecture, and hyper-parameters**:

1. **Manual (NumPy)** — implemented from scratch in
   [`rnn_scratch_multi_layer.py`](rnn_scratch_multi_layer.py).
2. **TensorFlow / Keras** — [`rnn_tensorflow.py`](rnn_tensorflow.py).
3. **PyTorch** — [`rnn_pytorch.py`](rnn_pytorch.py).

All three expose the same interface (`train()`, `predict()`), so the shared
helpers in [`utils.py`](utils.py) — `evaluate`, `generate` — and
[`compare.py`](compare.py)'s `compare_models` work on every model unchanged.

**Pipeline:** raw text → vocabulary → training sequences → train/test split →
train the three models → compare train/test accuracy → generate text.

| Section | Contents |
|---------|----------|
| 1 | Imports |
| 2 | Data |
| 3 | Character level — 3-way comparison |
| 4 | Word level — Word2Vec · GloVe · FastText · BERT (each a 3-way comparison) |

> **Note on the comparison.** The frameworks train with **Adam** while the
> from-scratch model uses plain **SGD + gradient clipping**, so their learned
> weights — and accuracies — differ. This is a realistic "library vs scratch"
> comparison, not a bit-for-bit match.

## 1. Imports

In [1]:
import importlib

import numpy as np

import rnn_scratch_multi_layer, rnn_tensorflow, rnn_pytorch, utils, compare

# Reload local modules so edits to the .py files are picked up without
# having to restart the kernel.
for _m in (rnn_scratch_multi_layer, rnn_tensorflow, rnn_pytorch, utils, compare):
    importlib.reload(_m)

from rnn_scratch_multi_layer import RNN
from rnn_tensorflow import KerasRNN
from rnn_pytorch import TorchRNN
from utils import generate_dataset, train_test_split, evaluate, predict_next, generate
from compare import compare_models

## 2. Data

A small block of text to learn from. With a corpus this size the model can only
learn surface statistics, but it is enough to demonstrate and compare the RNNs.

In [2]:
sentences = [
'Machine Learning (ML) is a branch of Artificial Intelligence (AI)',
'that enables computers to learn from data and improve their performance',
'without being explicitly programmed. ML algorithms identify patterns in',
'historical data and use those patterns to make predictions or decisions on new data.',
'Machine learning is widely used in various applications such as recommendation systems,',
'image recognition, speech processing, fraud detection, healthcare diagnostics,',
'and autonomous vehicles. The main types of machine learning are supervised learning,',
'unsupervised learning, and reinforcement learning. As the amount of available data continues to grow,',
'machine learning plays an increasingly important role in helping organizations automate tasks,',
'gain insights, and make data-driven decisions.'
]
sentence_words = [s.split() for s in sentences]
words = [w for s in sentence_words for w in s]
text = " ".join(words)

In [3]:
def train_models(X_train, Y_train, hidden_layers, lr, epochs, batch_size, task):

    print('-'*100)
    print('Training Manual RNN')
    manual_rnn = RNN(X_train, Y_train, hidden_layers=hidden_layers,
                     learning_rate=lr, epochs=epochs, batch_size=batch_size, task=task)
    manual_rnn.train()

    print()
    print('-'*100)
    print('Training TensorFlow RNN')
    keras_rnn = KerasRNN(X_train, Y_train, hidden_layers=hidden_layers,
                         learning_rate=lr, epochs=epochs, batch_size=batch_size, task=task).train()

    print()
    print('-'*100)
    print('Training PyTorch RNN')
    torch_rnn = TorchRNN(X_train, Y_train, hidden_layers=hidden_layers,
                         learning_rate=lr, epochs=epochs, batch_size=batch_size, task=task).train()

    print()
    print('-'*100)

    trained_models = {"manual (numpy)": manual_rnn,
                      "tensorflow": keras_rnn,
                      "pytorch": torch_rnn}

    return trained_models

## 3. Character-level Text Generation

Slide a window of length `T_x` over the text: each input `X` is a chunk of
characters and the target `Y` is that chunk shifted one character left, so the
model learns to predict the *next* character at every step. Arrays are one-hot
encoded into `(m, T_x, vocab_size)`.

In [4]:
# One-hot character sequences (word_vectors is unused in char mode).
char_X, char_Y, char_to_index, index_to_char = generate_dataset(
    text, T_x=10, is_char=True, word_vectors=[], seq_length=25
)
print(f"X: {char_X.shape}   Y: {char_Y.shape}   (m, T_x, vocab_size)")

# Hold out 20% of the sequences for testing.
X_train, X_test, Y_train, Y_test = train_test_split(char_X, char_Y, test_size=0.2)

Generated 754 training sequences of length 10 from a corpus of 790 tokens, with a vocabulary of 35 unique characters.
X: (754, 10, 35)   Y: (754, 10, 35)   (m, T_x, vocab_size)
Split 754 sequences -> 603 train / 151 test.


### 3.1 Train the three models

Same `hidden_layers` and hyper-parameters for all three, trained on the same
training split.

In [5]:
hidden_layers, lr, epochs, batch_size, task = (50,), 0.01, 15, 32, 'classification'
print('Character Models')
char_models = train_models(
    X_train, Y_train, hidden_layers, lr, epochs, batch_size, task
)

Character Models
----------------------------------------------------------------------------------------------------
Training Manual RNN
Epoch 1/15, Loss: 31.8050
Epoch 2/15, Loss: 28.9564
Epoch 3/15, Loss: 26.2527
Epoch 4/15, Loss: 24.3097
Epoch 5/15, Loss: 22.9777
Epoch 6/15, Loss: 21.6789
Epoch 7/15, Loss: 20.6966
Epoch 8/15, Loss: 19.6557
Epoch 9/15, Loss: 18.8327
Epoch 10/15, Loss: 17.7321
Epoch 11/15, Loss: 16.4005
Epoch 12/15, Loss: 15.4462
Epoch 13/15, Loss: 14.6960
Epoch 14/15, Loss: 13.8699
Epoch 15/15, Loss: 13.2253

----------------------------------------------------------------------------------------------------
Training TensorFlow RNN
[Keras] trained 15 epochs (batch_size=32), final loss 0.7266

----------------------------------------------------------------------------------------------------
Training PyTorch RNN
[PyTorch] epoch 1/15, loss 3.0327
[PyTorch] epoch 2/15, loss 2.6384
[PyTorch] epoch 3/15, loss 2.2272
[PyTorch] epoch 4/15, loss 1.9360
[PyTorch] epoch 5/15

### 3.2 Compare

`compare_models` scores each trained model on the train/test splits and
generates a sequence from the same seed character.

In [6]:
compare_models(
    char_models, X_train, Y_train, X_test, Y_test,
    embedding=char_to_index, decoder=index_to_char,
    seed_word='M', is_char=True, num_gen=100,
)

=== accuracy: train / test ===
  model               train     test
  manual (numpy)     0.6143   0.5225
  tensorflow         0.7891   0.7026
  pytorch            0.7881   0.6940

=== generation from 'M' ===
  manual (numpy)   ML algoriches to gromed is autons, suchiof Artions aplection, hearning, suchiof Artions aplection, he
  tensorflow       ML algorithms identify patterns toromoun vehicles. The main types of machine learning is widely used 
  pytorch          Machine learning, unsupervised learning predarning, unsupervised learning predarning, unsupervised le


## 4. Word-level Prediction

Now each input token is a dense **word embedding** instead of a one-hot
character; targets stay one-hot over the vocabulary, so the model still runs in
`task="classification"` and predicts the next word.

Each subsection feeds the same corpus through a different embedding
(Word2Vec · GloVe · FastText · BERT) and runs the **full 3-way comparison**
(manual / TensorFlow / PyTorch).

### 4.1 Word2Vec embeddings

Train a Word2Vec model on our sentences (skip-gram) and use its vectors as the RNN's input features. This section also demonstrates a **2-layer stack** (`hidden_layers=(100, 64)`).

In [7]:
from gensim.models import Word2Vec

w2v_model = Word2Vec(
    sentence_words,
    vector_size=100,  # dimensionality of the word vectors
    window=5,        # context window size
    min_count=1,     # minimum word frequency to include in the model
    workers=4,       # number of CPU cores to use for training
    sg=1             # use skip-gram architecture (sg=0 for CBOW)
)
word_vectors = w2v_model.wv

In [8]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, is_char=False, word_vectors=word_vectors)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (m, T_x, vector_size)")

X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 102 training sequences of length 5 from a corpus of 108 tokens, with a vocabulary of 88 unique words.
X: (102, 5, 100)   Y: (102, 5, 88)   (m, T_x, vector_size)
Split 102 sequences -> 82 train / 20 test.


In [19]:
hidden_layers, lr, epochs, batch_size, task = (100, 64), 0.01, 10, 32, 'classification'
print('Word2Vec Models')
w2v_models = train_models(
    X_train, Y_train, hidden_layers, lr, epochs, batch_size, task
)

Word2Vec Models
----------------------------------------------------------------------------------------------------
Training Manual RNN
Epoch 1/10, Loss: 22.3767
Epoch 2/10, Loss: 22.3161
Epoch 3/10, Loss: 22.2377
Epoch 4/10, Loss: 22.1836
Epoch 5/10, Loss: 22.1222
Epoch 6/10, Loss: 22.0796
Epoch 7/10, Loss: 22.0326
Epoch 8/10, Loss: 21.9951
Epoch 9/10, Loss: 21.9817
Epoch 10/10, Loss: 21.9534

----------------------------------------------------------------------------------------------------
Training TensorFlow RNN
[Keras] trained 10 epochs (batch_size=32), final loss 1.7674

----------------------------------------------------------------------------------------------------
Training PyTorch RNN
[PyTorch] epoch 1/10, loss 4.4737
[PyTorch] epoch 2/10, loss 4.3915
[PyTorch] epoch 3/10, loss 4.3628
[PyTorch] epoch 4/10, loss 4.3254
[PyTorch] epoch 5/10, loss 4.3133
[PyTorch] epoch 6/10, loss 4.2727
[PyTorch] epoch 7/10, loss 4.2446
[PyTorch] epoch 8/10, loss 4.1371
[PyTorch] epoch 9/10

In [20]:
compare_models(
    w2v_models, X_train, Y_train, X_test, Y_test,
    embedding=word_vectors, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual (numpy)     0.0463   0.0400
  tensorflow         0.6439   0.2000
  pytorch            0.0829   0.0200

=== generation from 'Machine' ===
  manual (numpy)   Machine insights, in patterns improve detection, (AI) patterns in The branch
  tensorflow       Machine Intelligence patterns is such use of plays learning healthcare without
  pytorch          Machine algorithms learning learning, data. of Intelligence tasks, learning data learning,


### 4.2 GloVe — pre-trained embeddings (Google-News word2vec)

Swap our small in-house vectors for Google's pre-trained 300-d word2vec vectors (trained on ~100B words of Google News).

In [21]:
# pre-trained vectors from the Google-News dataset (300d, 3M words)
import gensim.downloader as api
glove = api.load("word2vec-google-news-300")

In [22]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, is_char=False, word_vectors=glove)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (m, T_x, vector_size)")

X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 72 training sequences of length 5 from a corpus of 78 tokens, with a vocabulary of 67 unique words.
X: (72, 5, 300)   Y: (72, 5, 67)   (m, T_x, vector_size)
Split 72 sequences -> 58 train / 14 test.


In [24]:
hidden_layers, lr, epochs, batch_size, task = (100,), 0.02, 15, 32, 'classification'
print('GloVe Models')
glove_models = train_models(
    X_train, Y_train, hidden_layers, lr, epochs, batch_size, task
)

GloVe Models
----------------------------------------------------------------------------------------------------
Training Manual RNN
Epoch 1/15, Loss: 21.0283
Epoch 2/15, Loss: 20.8501
Epoch 3/15, Loss: 20.6904
Epoch 4/15, Loss: 20.4577
Epoch 5/15, Loss: 20.0070
Epoch 6/15, Loss: 19.2526
Epoch 7/15, Loss: 18.1849
Epoch 8/15, Loss: 16.8439
Epoch 9/15, Loss: 14.8460
Epoch 10/15, Loss: 12.7629
Epoch 11/15, Loss: 10.5374
Epoch 12/15, Loss: 8.4137
Epoch 13/15, Loss: 6.5226
Epoch 14/15, Loss: 5.0134
Epoch 15/15, Loss: 3.9791

----------------------------------------------------------------------------------------------------
Training TensorFlow RNN
[Keras] trained 15 epochs (batch_size=32), final loss 0.0387

----------------------------------------------------------------------------------------------------
Training PyTorch RNN
[PyTorch] epoch 1/15, loss 3.9920
[PyTorch] epoch 2/15, loss 2.4795
[PyTorch] epoch 3/15, loss 1.0545
[PyTorch] epoch 4/15, loss 0.4016
[PyTorch] epoch 5/15, loss 0

In [25]:
compare_models(
    glove_models, X_train, Y_train, X_test, Y_test,
    embedding=glove, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual (numpy)     0.9000   0.8571
  tensorflow         0.9759   0.9429
  pytorch            0.9759   0.9571

=== generation from 'Machine' ===
  manual (numpy)   Machine learning plays an increasingly important role in role in various
  tensorflow       Machine learning is widely used in various applications such as recommendation
  pytorch          Machine learning is widely used in various applications such as recommendation


### 4.3 FastText embeddings

FastText builds word vectors from character n-grams, so it can embed even rare or out-of-vocabulary words. Trained here on our own sentences.

In [26]:
from gensim.models import FastText

ft_model = FastText(
    sentence_words,
    vector_size=100,  # dimensionality of the word vectors
    window=5,        # context window size
    min_count=1,     # minimum word frequency to include in the model
    workers=4,       # number of CPU cores to use for training
    sg=1             # use skip-gram architecture (sg=0 for CBOW)
)
fasttext_vectors = ft_model.wv

In [27]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, is_char=False, word_vectors=fasttext_vectors)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (m, T_x, vector_size)")

X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 102 training sequences of length 5 from a corpus of 108 tokens, with a vocabulary of 88 unique words.
X: (102, 5, 100)   Y: (102, 5, 88)   (m, T_x, vector_size)
Split 102 sequences -> 82 train / 20 test.


In [40]:
hidden_layers, lr, epochs, batch_size, task = (50,), 0.009, 15, 32, 'classification'
print('FastText Models')
fasttext_models = train_models(
    X_train, Y_train, hidden_layers, lr, epochs, batch_size, task
)

FastText Models
----------------------------------------------------------------------------------------------------
Training Manual RNN
Epoch 1/15, Loss: 22.3778
Epoch 2/15, Loss: 22.3126
Epoch 3/15, Loss: 22.2433
Epoch 4/15, Loss: 22.1944
Epoch 5/15, Loss: 22.1290
Epoch 6/15, Loss: 22.0986
Epoch 7/15, Loss: 22.0422
Epoch 8/15, Loss: 22.0231
Epoch 9/15, Loss: 21.9886
Epoch 10/15, Loss: 21.9268
Epoch 11/15, Loss: 21.8795
Epoch 12/15, Loss: 21.8914
Epoch 13/15, Loss: 21.8779
Epoch 14/15, Loss: 21.8364
Epoch 15/15, Loss: 21.8508

----------------------------------------------------------------------------------------------------
Training TensorFlow RNN
[Keras] trained 15 epochs (batch_size=32), final loss 4.0832

----------------------------------------------------------------------------------------------------
Training PyTorch RNN
[PyTorch] epoch 1/15, loss 4.4876
[PyTorch] epoch 2/15, loss 4.4300
[PyTorch] epoch 3/15, loss 4.3612
[PyTorch] epoch 4/15, loss 4.3640
[PyTorch] epoch 5/15,

In [41]:
compare_models(
    fasttext_models, X_train, Y_train, X_test, Y_test,
    embedding=fasttext_vectors, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual (numpy)     0.0463   0.0400
  tensorflow         0.0780   0.0400
  pytorch            0.0463   0.0200

=== generation from 'Machine' ===
  manual (numpy)   Machine to of algorithms As reinforcement such available role role is
  tensorflow       Machine main and improve is of insights, organizations data. various learning,
  pytorch          Machine image explicitly from diagnostics, branch the used processing, are computers


### 4.4 BERT embeddings

Use a pre-trained BERT model to produce a contextual `[CLS]` embedding per word, then feed those 768-d vectors to the RNNs.

In [42]:
from transformers import BertTokenizer, BertModel
import torch

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_model = BertModel.from_pretrained('bert-base-uncased')

bert_word_to_vec = {}
for word in words:
    inputs = tokenizer(word, return_tensors="pt")
    with torch.no_grad():
        outputs = bert_model(**inputs)
    bert_word_to_vec[word] = outputs.last_hidden_state[:, 0, :].squeeze().numpy()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [43]:
input_seqs, target_seqs, vocab_to_index, index_to_vocab = generate_dataset(
    words, T_x=5, is_char=False, word_vectors=bert_word_to_vec)
print(f"X: {input_seqs.shape}   Y: {target_seqs.shape}   (m, T_x, vector_size)")

X_train, X_test, Y_train, Y_test = train_test_split(input_seqs, target_seqs, test_size=0.2)

Generated 102 training sequences of length 5 from a corpus of 108 tokens, with a vocabulary of 88 unique words.
X: (102, 5, 768)   Y: (102, 5, 88)   (m, T_x, vector_size)
Split 102 sequences -> 82 train / 20 test.


In [56]:
hidden_layers, lr, epochs, batch_size, task = (50,), 0.01255, 15, 32, 'classification'
print('BERT Models')
bert_models = train_models(
    X_train, Y_train, hidden_layers, lr, epochs, batch_size, task
)

BERT Models
----------------------------------------------------------------------------------------------------
Training Manual RNN
Epoch 1/15, Loss: 22.3280
Epoch 2/15, Loss: 22.2148
Epoch 3/15, Loss: 21.8721
Epoch 4/15, Loss: 21.4830
Epoch 5/15, Loss: 22.2520
Epoch 6/15, Loss: 21.0068
Epoch 7/15, Loss: 21.3943
Epoch 8/15, Loss: 20.6910
Epoch 9/15, Loss: 19.9431
Epoch 10/15, Loss: 19.9136
Epoch 11/15, Loss: 19.2937
Epoch 12/15, Loss: 18.8132
Epoch 13/15, Loss: 18.6644
Epoch 14/15, Loss: 18.1181
Epoch 15/15, Loss: 16.4618

----------------------------------------------------------------------------------------------------
Training TensorFlow RNN
[Keras] trained 15 epochs (batch_size=32), final loss 0.1919

----------------------------------------------------------------------------------------------------
Training PyTorch RNN
[PyTorch] epoch 1/15, loss 4.4273
[PyTorch] epoch 2/15, loss 3.9087
[PyTorch] epoch 3/15, loss 3.3542
[PyTorch] epoch 4/15, loss 2.8332
[PyTorch] epoch 5/15, los

In [57]:
compare_models(
    bert_models, X_train, Y_train, X_test, Y_test,
    embedding=bert_word_to_vec, decoder=index_to_vocab,
    seed_word='Machine', num_gen=10, sample=True,
)

=== accuracy: train / test ===
  model               train     test
  manual (numpy)     0.1780   0.0700
  tensorflow         0.9439   0.8800
  pytorch            0.9390   0.8600

=== generation from 'Machine' ===
  manual (numpy)   Machine learning without explicitly are important learning of supervised Machine learning
  tensorflow       Machine learning is widely used in various applications such as recommendation
  pytorch          Machine Learning plays an increasingly important role in historical data and
